In [1]:
### Superstructure Optimization — Gurobi version
import numpy as np
import sys
import time
from itertools import combinations

try:
    import gurobipy as gp
    from gurobipy import GRB, quicksum
except ImportError:
    sys.exit(1)

In [2]:
# OCFE setup
# We discretize each membrane element using Gauss-Lobatto-Legendre (GLL) points.
# These are optimal for polynomial collocation because they include both endpoints
# and cluster points near the boundaries where gradients are steepest.
# The 5-point GLL set on [-1,1] maps to [0,1] via s = (p+1)/2.
# The differentiation matrix D encodes the Lagrange polynomial derivatives:
#   D[j,k] = d(phi_k)/ds evaluated at s_j
# This turns the membrane ODE (dF/dA = -J) into an algebraic system the
# optimizer can handle directly — no time-stepping needed inside the solver.

def build_ocfe_matrices():
    p = np.array([-1, -np.sqrt(3/7), 0, np.sqrt(3/7), 1], dtype=float)
    s = (p + 1) / 2

    n = len(s)
    D = np.zeros((n, n))

    for j in range(n):
        for k in range(n):
            if j == k:
                D[j, k] = sum(1.0 / (s[k] - s[m]) for m in range(n) if m != k)
            else:
                num = 1.0
                den = 1.0
                for m in range(n):
                    if m not in (j, k):
                        num *= s[j] - s[m]
                    if m != k:
                        den *= s[k] - s[m]
                D[j, k] = num / den
    return s, D

S, DM = build_ocfe_matrices()

In [3]:
# Discretization parameters
NEL = 2    # finite elements per module
NCP = 4    # interior collocation points per element
NPT = 5    # total points per element including boundaries
NMOD = 5   # max modules in superstructure

# Physical constants
GPU = 3.34e-10 * 1e-3 * 3600 * 1e5   # 1 GPU in kmol/(m2 h bar)
# Component indices: 1=N2, 2=O2
PI = {1: 80   * GPU,   
      2: 1600 * GPU}   
PR = 10.0
PP = 1.0
# Feed specifications
FT = 2300.0
XF = {1: 0.79, 2: 0.21}
FF = {i: FT * XF[i] for i in (1, 2)}
# Cost parameters
CA  = 20.0      # $/m2/h
PHI = 1000.0    # $/module/h
# Bounds
AMAX = 5000.0
AMIN = 100.0
FMAX = FT * 5
EPS  = 1e-4
# Big-M values
BIGM_FLUX = 60.0
BIGM_MB   = 2e5
COMPS = (1, 2)
MODS  = tuple(range(1, NMOD + 1))
ELS   = tuple(range(1, NEL + 1))
PTS   = tuple(range(NPT))
PTSi  = tuple(range(1, NPT))

In [4]:
# Model build
def build_model():
    m = gp.Model("AirSep")
    m.Params.OutputFlag = 0          
    m.Params.NonConvex  = 2         

    V = {}   # container for all decision variables (so we can reach them later)

    # Topology binaries
    V['y']   = m.addVars(MODS,                vtype=GRB.BINARY, name="y")
    V['zIR'] = m.addVars([1], MODS,           vtype=GRB.BINARY, name="zIR")
    V['zRR'] = m.addVars(MODS, MODS,          vtype=GRB.BINARY, name="zRR")
    V['zSR'] = m.addVars(MODS, MODS,          vtype=GRB.BINARY, name="zSR")
    V['zRP'] = m.addVars(MODS, [1],           vtype=GRB.BINARY, name="zRP")
    V['zSP'] = m.addVars(MODS, [1],           vtype=GRB.BINARY, name="zSP")

    # Areas
    V['A']   = m.addVars(MODS, lb=0.0, ub=AMAX,        name="A")
    V['dAe'] = m.addVars(MODS, lb=0.0, ub=AMAX / NEL,  name="dAe")

    # Split fractions
    V['split_feed'] = m.addVars([1], MODS,        lb=0.0, ub=1.0, name="split_feed")
    V['split_RR']   = m.addVars(MODS, MODS,       lb=0.0, ub=1.0, name="split_RR")
    V['split_RP']   = m.addVars(MODS, [1],        lb=0.0, ub=1.0, name="split_RP")
    V['split_SR']   = m.addVars(MODS, MODS,       lb=0.0, ub=1.0, name="split_SR")
    V['split_SP']   = m.addVars(MODS, [1],        lb=0.0, ub=1.0, name="split_SP")

    # Network flows
    V['FIR'] = m.addVars(COMPS, [1], MODS,            lb=0.0, ub=FMAX, name="FIR")
    V['FRI'] = m.addVars(COMPS, MODS,                 lb=0.0, ub=FMAX, name="FRI")
    V['FRO'] = m.addVars(COMPS, MODS,                 lb=0.0, ub=FMAX, name="FRO")
    V['FSI'] = m.addVars(COMPS, MODS,                 lb=0.0, ub=FMAX, name="FSI")
    V['FSO'] = m.addVars(COMPS, MODS,                 lb=0.0, ub=FMAX, name="FSO")
    V['FRR'] = m.addVars(COMPS, MODS, MODS,           lb=0.0, ub=FMAX, name="FRR")
    V['FSR'] = m.addVars(COMPS, MODS, MODS,           lb=0.0, ub=FMAX, name="FSR")
    V['FRP'] = m.addVars(COMPS, MODS, [1],            lb=0.0, ub=FMAX, name="FRP")
    V['FSP'] = m.addVars(COMPS, MODS, [1],            lb=0.0, ub=FMAX, name="FSP")
    V['FPR'] = m.addVars(COMPS, [1],                  lb=0.0, ub=FMAX, name="FPR")
    V['FPS'] = m.addVars(COMPS, [1],                  lb=0.0, ub=FMAX, name="FPS")

    # OCFE profile variables
    V['FR']  = m.addVars(COMPS, MODS, ELS, PTS, lb=0.0,  ub=FMAX, name="FR")
    V['FS']  = m.addVars(COMPS, MODS, ELS, PTS, lb=0.0,  ub=FMAX, name="FS")
    V['FRt'] = m.addVars(MODS,  ELS, PTS,       lb=0.0,  ub=FMAX, name="FRt")
    V['FSt'] = m.addVars(MODS,  ELS, PTS,       lb=0.0,  ub=FMAX, name="FSt")
    V['xR']  = m.addVars(COMPS, MODS, ELS, PTS, lb=0.0,  ub=1.0,  name="xR")
    V['xS']  = m.addVars(COMPS, MODS, ELS, PTS, lb=0.0,  ub=1.0,  name="xS")
    V['Jf']  = m.addVars(COMPS, MODS, ELS, PTS, lb=0.0,  ub=50.0, name="Jf")

    m.update()

    # short aliases
    y, zIR, zRR, zSR, zRP, zSP        = V['y'], V['zIR'], V['zRR'], V['zSR'], V['zRP'], V['zSP']
    A, dAe                            = V['A'], V['dAe']
    split_feed                        = V['split_feed']
    split_RR, split_RP, split_SR, split_SP = V['split_RR'], V['split_RP'], V['split_SR'], V['split_SP']
    FIR, FRI, FRO, FSI, FSO           = V['FIR'], V['FRI'], V['FRO'], V['FSI'], V['FSO']
    FRR, FSR, FRP, FSP                = V['FRR'], V['FSR'], V['FRP'], V['FSP']
    FPR, FPS                          = V['FPR'], V['FPS']
    FR, FS, FRt, FSt, xR, xS, Jf      = V['FR'], V['FS'], V['FRt'], V['FSt'], V['xR'], V['xS'], V['Jf']

    # Objective
    m.setObjective(
        quicksum(CA * A[md] for md in MODS)
        + PHI * quicksum(y[md] for md in MODS),
        GRB.MINIMIZE)

    # Superstructure constraints
    for md in MODS:
        m.addConstr(dAe[md] == A[md] / NEL,           name=f"c_dA_{md}")
        m.addConstr(A[md]   <= y[md] * AMAX,          name=f"c_Au_{md}")
        m.addConstr(A[md]   >= y[md] * AMIN,          name=f"c_Al_{md}")

    # Feed split
    for i in COMPS:
        for nr in (1,):
            for md in MODS:
                m.addConstr(FIR[i, nr, md] == split_feed[nr, md] * FF[i],
                            name=f"c_fs_comp_{i}_{nr}_{md}")
    for nr in (1,):
        m.addConstr(quicksum(split_feed[nr, md] for md in MODS) == 1.0,
                    name=f"c_fs_total_{nr}")

    # Module mass-balance: inlet retentate = feed in + recycles in
    for i in COMPS:
        for md in MODS:
            m.addConstr(
                FRI[i, md] ==
                  quicksum(FIR[i, nr, md] for nr in (1,))
                + quicksum(FRR[i, mp, md] for mp in MODS if mp != md)
                + quicksum(FSR[i, mp, md] for mp in MODS if mp != md),
                name=f"c_rm_{i}_{md}")

            # No permeate-side feed at the inlet
            m.addConstr(FSI[i, md] == 0, name=f"c_si_{i}_{md}")

            # Outlet retentate splits to other modules + product
            m.addConstr(
                FRO[i, md] ==
                  quicksum(FRR[i, md, mp] for mp in MODS if mp != md)
                + quicksum(FRP[i, md, pr] for pr in (1,)),
                name=f"c_rs_{i}_{md}")

            # Outlet permeate splits to other modules + waste
            m.addConstr(
                FSO[i, md] ==
                  quicksum(FSP[i, md, ps] for ps in (1,))
                + quicksum(FSR[i, md, mp] for mp in MODS if mp != md),
                name=f"c_ps_{i}_{md}")

    # Composition-preserving splits
    for i in COMPS:
        for md in MODS:
            for mp in MODS:
                if md == mp:
                    continue
                m.addQConstr(FRR[i, md, mp] == split_RR[md, mp] * FRO[i, md],
                             name=f"c_comp_RR_{i}_{md}_{mp}")
                m.addQConstr(FSR[i, md, mp] == split_SR[md, mp] * FSO[i, md],
                             name=f"c_comp_SR_{i}_{md}_{mp}")
            for pr in (1,):
                m.addQConstr(FRP[i, md, pr] == split_RP[md, pr] * FRO[i, md],
                             name=f"c_comp_RP_{i}_{md}_{pr}")
            for ps in (1,):
                m.addQConstr(FSP[i, md, ps] == split_SP[md, ps] * FSO[i, md],
                             name=f"c_comp_SP_{i}_{md}_{ps}")

    # Splits must sum to 1
    for md in MODS:
        m.addConstr(
            quicksum(split_RR[md, mp] for mp in MODS if mp != md)
            + quicksum(split_RP[md, pr] for pr in (1,)) == 1.0,
            name=f"c_sum_split_R_{md}")
        m.addConstr(
            quicksum(split_SR[md, mp] for mp in MODS if mp != md)
            + quicksum(split_SP[md, ps] for ps in (1,)) == 1.0,
            name=f"c_sum_split_S_{md}")

    # Product / waste collectors
    for i in COMPS:
        for pr in (1,):
            m.addConstr(FPR[i, pr] == quicksum(FRP[i, md, pr] for md in MODS),
                        name=f"c_rp_{i}_{pr}")
        for ps in (1,):
            m.addConstr(FPS[i, ps] == quicksum(FSP[i, md, ps] for md in MODS),
                        name=f"c_pp_{i}_{ps}")

    # Big-M on network flows
    for i in COMPS:
        for nr in (1,):
            for md in MODS:
                m.addConstr(FIR[i, nr, md] <= zIR[nr, md] * FMAX,
                            name=f"c_bI_{i}_{nr}_{md}")
        for md in MODS:
            for mp in MODS:
                if md == mp:
                    continue
                m.addConstr(FRR[i, md, mp] <= zRR[md, mp] * FMAX,
                            name=f"c_bR_{i}_{md}_{mp}")
                m.addConstr(FSR[i, md, mp] <= zSR[md, mp] * FMAX,
                            name=f"c_bSR_{i}_{md}_{mp}")
            for pr in (1,):
                m.addConstr(FRP[i, md, pr] <= zRP[md, pr] * FMAX,
                            name=f"c_bP_{i}_{md}_{pr}")
            for ps in (1,):
                m.addConstr(FSP[i, md, ps] <= zSP[md, ps] * FMAX,
                            name=f"c_bS_{i}_{md}_{ps}")

    # Logic constraints
    for nr in (1,):
        for md in MODS:
            m.addConstr(zIR[nr, md] <= y[md], name=f"c_l1_{nr}_{md}")
    for md in MODS:
        for mp in MODS:
            if md == mp:
                continue
            m.addConstr(zRR[md, mp] <= y[md], name=f"c_l2_{md}_{mp}")
            m.addConstr(zRR[md, mp] <= y[mp], name=f"c_l3_{md}_{mp}")
            m.addConstr(zSR[md, mp] <= y[md], name=f"c_l4a_{md}_{mp}")
            m.addConstr(zSR[md, mp] <= y[mp], name=f"c_l4b_{md}_{mp}")
        for pr in (1,):
            m.addConstr(zRP[md, pr] <= y[md], name=f"c_l5_{md}_{pr}")
        for ps in (1,):
            m.addConstr(zSP[md, ps] <= y[md], name=f"c_l6_{md}_{ps}")

    # No self-loops
    for md in MODS:
        m.addConstr(zRR[md, md] == 0, name=f"c_nsR_{md}")
        m.addConstr(zSR[md, md] == 0, name=f"c_nsSR_{md}")
        for i in COMPS:
            m.addConstr(FRR[i, md, md] == 0, name=f"c_nfR_{i}_{md}")
            m.addConstr(FSR[i, md, md] == 0, name=f"c_nfSR_{i}_{md}")

    # Inactive modules must carry zero flow
    for i in COMPS:
        for md in MODS:
            m.addConstr(FRI[i, md] <= y[md] * FMAX, name=f"c_bRI_{i}_{md}")
            m.addConstr(FRO[i, md] <= y[md] * FMAX, name=f"c_bRO_{i}_{md}")
            m.addConstr(FSO[i, md] <= y[md] * FMAX, name=f"c_bSO_{i}_{md}")

    # At least one module active
    m.addConstr(quicksum(y[md] for md in MODS) >= 1, name="c_min")

    # If active, must have at least one outlet on each side
    for md in MODS:
        m.addConstr(
            quicksum(zRP[md, pr] for pr in (1,))
            + quicksum(zRR[md, mp] for mp in MODS if mp != md) >= y[md],
            name=f"c_ro_{md}")
        m.addConstr(
            quicksum(zSP[md, ps] for ps in (1,))
            + quicksum(zSR[md, mp] for mp in MODS if mp != md) >= y[md],
            name=f"c_po_{md}")

    # OCFE constraints
    # Zero-out OCFE variables when module inactive
    for md in MODS:
        for l in ELS:
            for j in PTS:
                for i in COMPS:
                    m.addConstr(FR[i, md, l, j] <= y[md] * FMAX,
                                name=f"c_bFR_{i}_{md}_{l}_{j}")
                    m.addConstr(FS[i, md, l, j] <= y[md] * FMAX,
                                name=f"c_bFS_{i}_{md}_{l}_{j}")
                    m.addConstr(Jf[i, md, l, j] <= y[md] * 50.0,
                                name=f"c_bJf_{i}_{md}_{l}_{j}")
                m.addConstr(FRt[md, l, j] <= y[md] * FMAX + EPS,
                            name=f"c_bTR_{md}_{l}_{j}")
                m.addConstr(FSt[md, l, j] <= y[md] * FMAX + EPS,
                            name=f"c_bTS_{md}_{l}_{j}")

    # Boundary conditions
    for i in COMPS:
        for md in MODS:
            m.addConstr(FR[i, md, 1, 0] == FRI[i, md], name=f"c_rI_{i}_{md}")
            m.addConstr(FS[i, md, 1, 0] == FSI[i, md], name=f"c_sI_{i}_{md}")
            m.addConstr(FRO[i, md] == FR[i, md, NEL, NCP], name=f"c_rO_{i}_{md}")
            m.addConstr(FSO[i, md] == FS[i, md, NEL, NCP], name=f"c_sO_{i}_{md}")

    # Element continuity
    for i in COMPS:
        for md in MODS:
            for l in ELS:
                if l == 1:
                    continue
                m.addConstr(FR[i, md, l, 0] == FR[i, md, l - 1, NCP],
                            name=f"c_ecR_{i}_{md}_{l}")
                m.addConstr(FS[i, md, l, 0] == FS[i, md, l - 1, NCP],
                            name=f"c_ecS_{i}_{md}_{l}")

    # Total flows at every collocation point
    for md in MODS:
        for l in ELS:
            for j in PTS:
                m.addConstr(FRt[md, l, j] == quicksum(FR[i, md, l, j] for i in COMPS),
                            name=f"c_tR_{md}_{l}_{j}")
                m.addConstr(FSt[md, l, j] == quicksum(FS[i, md, l, j] for i in COMPS),
                            name=f"c_tS_{md}_{l}_{j}")

    # Mole fractions
    for i in COMPS:
        for md in MODS:
            for l in ELS:
                for j in PTS:
                    m.addQConstr(xR[i, md, l, j] * FRt[md, l, j] == FR[i, md, l, j],
                                 name=f"c_xR_{i}_{md}_{l}_{j}")
                    m.addQConstr(xS[i, md, l, j] * FSt[md, l, j] == FS[i, md, l, j],
                                 name=f"c_xS_{i}_{md}_{l}_{j}")

    # Flux model with big-M deactivation
    for i in COMPS:
        for md in MODS:
            for l in ELS:
                for j in PTS:
                    expr = Jf[i, md, l, j] - PI[i] * (xR[i, md, l, j] * PR
                                                       - xS[i, md, l, j] * PP)
                    m.addConstr(expr >= -BIGM_FLUX * (1 - y[md]),
                                name=f"c_flux_lo_{i}_{md}_{l}_{j}")
                    m.addConstr(expr <=  BIGM_FLUX * (1 - y[md]),
                                name=f"c_flux_hi_{i}_{md}_{l}_{j}")

    # OCFE mass balances
    DM_dict = {(j, k): float(DM[j, k]) for j in PTS for k in PTS}
    for i in COMPS:
        for md in MODS:
            for l in ELS:
                for j in PTSi:
                    expr_R = (quicksum(DM_dict[j, k] * FR[i, md, l, k] for k in PTS)
                              + Jf[i, md, l, j] * dAe[md])
                    expr_S = (quicksum(DM_dict[j, k] * FS[i, md, l, k] for k in PTS)
                              - Jf[i, md, l, j] * dAe[md])
                    m.addQConstr(expr_R >= -BIGM_MB * (1 - y[md]),
                                 name=f"c_mbR_lo_{i}_{md}_{l}_{j}")
                    m.addQConstr(expr_R <=  BIGM_MB * (1 - y[md]),
                                 name=f"c_mbR_hi_{i}_{md}_{l}_{j}")
                    m.addQConstr(expr_S >= -BIGM_MB * (1 - y[md]),
                                 name=f"c_mbS_lo_{i}_{md}_{l}_{j}")
                    m.addQConstr(expr_S <=  BIGM_MB * (1 - y[md]),
                                 name=f"c_mbS_hi_{i}_{md}_{l}_{j}")

    # Product specs
    m.addConstr(FPR[1, 1] >= 0.98 * (FPR[1, 1] + FPR[2, 1]), name="c_pur")
    m.addConstr(FPR[1, 1] >= 0.50 * FF[1],                   name="c_rec")

    m.update()
    m._V = V       
    return m

In [5]:
def ren_perm_out_flows(fN, fO, A, ns=60):
    if A < 1:
        return float(fN), float(fO), 0.0, 0.0
    y = np.array([float(fN), float(fO), 0.0, 0.0])
    dA = A / ns
    def rhs(state):
        rN, rO, pN, pO = state
        Ft_r = rN + rO
        Ft_p = pN + pO
        if Ft_r < 1e-12:
            return np.zeros(4)
        xNr = rN / Ft_r
        xOr = rO / Ft_r
        if Ft_p > 1e-12:
            xNp = pN / Ft_p
            xOp = pO / Ft_p
        else:
            xOp = 0.8
            xNp = 0.2
        JN = PI[1] * max(xNr * PR - xNp * PP, 0.0)
        JO = PI[2] * max(xOr * PR - xOp * PP, 0.0)
        return np.array([-JN, -JO, JN, JO])
    for _ in range(ns):
        k1 = rhs(y)
        k2 = rhs(y + 0.5 * dA * k1)
        k3 = rhs(y + 0.5 * dA * k2)
        k4 = rhs(y + dA * k3)
        y = y + (dA / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        y = np.maximum(y, 0.0)
    return float(y[0]), float(y[1]), float(y[2]), float(y[3])
def sim_net(active, feed_to, rr, sr, rp, sp, areas):
    rec = {}
    flows = {}
    for iteration in range(200):
        flows = {}
        for md in active:
            inN = FF[1] / len(feed_to) if md in feed_to else 0.0
            inO = FF[2] / len(feed_to) if md in feed_to else 0.0
            for src in active:
                if src == md:
                    continue
                if (src, md) in rr:
                    inN += rec.get((1, src, md), 0.0)
                    inO += rec.get((2, src, md), 0.0)
                if (src, md) in sr:
                    inN += rec.get((1, src, md), 0.0)
                    inO += rec.get((2, src, md), 0.0)
            rN, rO, pN, pO = ren_perm_out_flows(max(inN, EPS), max(inO, EPS),
                                                areas.get(md, 2000))
            flows[md] = {'inN': inN, 'inO': inO,
                         'rN': rN,  'rO': rO,
                         'pN': pN,  'pO': pO}
        new_rec = {}
        max_diff = 0.0
        for src in active:
            f = flows[src]
            for dst in active:
                if dst == src:
                    continue
                if (src, dst) in rr:
                    n_out = (sum(1 for d in active if d != src and (src, d) in rr)
                             + (1 if (src, 1) in rp else 0))
                    frac = 1.0 / max(n_out, 1)
                    new_rec[(1, src, dst)] = f['rN'] * frac
                    new_rec[(2, src, dst)] = f['rO'] * frac
                if (src, dst) in sr:
                    n_out = (sum(1 for d in active if d != src and (src, d) in sr)
                             + (1 if (src, 1) in sp else 0))
                    frac = 1.0 / max(n_out, 1)
                    new_rec[(1, src, dst)] = f['pN'] * frac
                    new_rec[(2, src, dst)] = f['pO'] * frac
        for key in new_rec:
            max_diff = max(max_diff, abs(new_rec[key] - rec.get(key, 0.0)))
            rec[key] = rec.get(key, 0.0) * 0.5 + new_rec[key] * 0.5
        if max_diff < 0.1 and iteration > 5:
            break
    prodN = prodO = 0.0
    for md in active:
        if (md, 1) in rp:
            f = flows[md]
            n_out = (sum(1 for d in active if d != md and (md, d) in rr)
                     + (1 if (md, 1) in rp else 0))
            frac = 1.0 / max(n_out, 1)
            prodN += f['rN'] * frac
            prodO += f['rO'] * frac
    pt = prodN + prodO
    if pt < 1.0:
        return None
    return {'flows': flows, 'pur': prodN / pt, 'rec': prodN / FF[1], 'recycles': rec}
def _set(var, val):
    try:
        var.Start = float(val)
    except Exception:
        pass
def _fix(var, val):
    var.LB = float(val)
    var.UB = float(val)

In [6]:
def init(m, active, feed_to, rr, sr, rp, sp, areas):
    V = m._V
    sim_result = sim_net(active, feed_to, rr, sr, rp, sp, areas)
    flows = sim_result['flows'] if sim_result else {}
    # topology binaries: set start AND fix
    for md in MODS:
        is_active = md in active
        _set(V['y'][md], 1 if is_active else 0)
        _fix(V['y'][md], 1 if is_active else 0)
        _set(V['zIR'][1, md], 1 if md in feed_to else 0)
        _fix(V['zIR'][1, md], 1 if md in feed_to else 0)
        _set(V['zRP'][md, 1], 1 if (md, 1) in rp else 0)
        _fix(V['zRP'][md, 1], 1 if (md, 1) in rp else 0)
        _set(V['zSP'][md, 1], 1 if (md, 1) in sp else 0)
        _fix(V['zSP'][md, 1], 1 if (md, 1) in sp else 0)
        for mp in MODS:
            v_rr = 1 if (md, mp) in rr else 0
            v_sr = 1 if (md, mp) in sr else 0
            _set(V['zRR'][md, mp], v_rr); _fix(V['zRR'][md, mp], v_rr)
            _set(V['zSR'][md, mp], v_sr); _fix(V['zSR'][md, mp], v_sr)
    # continuous warm starts
    for md in MODS:
        if md not in active:
            _set(V['A'][md], 0.0)
            _set(V['dAe'][md], 0.0)
            for i in COMPS:
                for v in [V['FIR'][i, 1, md], V['FRI'][i, md], V['FRO'][i, md],
                          V['FSI'][i, md], V['FSO'][i, md],
                          V['FRP'][i, md, 1], V['FSP'][i, md, 1]]:
                    _set(v, 0.0)
                for mp in MODS:
                    _set(V['FRR'][i, md, mp], 0.0)
                    _set(V['FSR'][i, md, mp], 0.0)
            for l in ELS:
                for j in PTS:
                    for i in COMPS:
                        _set(V['FR'][i, md, l, j], 0.0)
                        _set(V['FS'][i, md, l, j], 0.0)
                        _set(V['Jf'][i, md, l, j], 0.0)
                        _set(V['xR'][i, md, l, j], 0.5)
                        _set(V['xS'][i, md, l, j], 0.5)
                    _set(V['FRt'][md, l, j], 0.0)
                    _set(V['FSt'][md, l, j], 0.0)
            continue
        f = flows.get(md, {
            'inN': FF[1], 'inO': FF[2],
            'rN':  FF[1] * 0.7, 'rO': FF[2] * 0.3,
            'pN':  FF[1] * 0.3, 'pO': FF[2] * 0.7})
        A_md = areas.get(md, 2000)
        _set(V['A'][md],   A_md)
        _set(V['dAe'][md], A_md / NEL)
        _set(V['split_feed'][1, md],
             1.0 / len(feed_to) if md in feed_to else 0.0)
        for i in COMPS:
            _set(V['FIR'][i, 1, md], FF[i] / len(feed_to) if md in feed_to else 0.0)
        _set(V['FRI'][1, md], max(f['inN'], EPS))
        _set(V['FRI'][2, md], max(f['inO'], EPS))
        _set(V['FRO'][1, md], max(f['rN'],  EPS))
        _set(V['FRO'][2, md], max(f['rO'],  EPS))
        _set(V['FSO'][1, md], max(f['pN'],  EPS))
        _set(V['FSO'][2, md], max(f['pO'],  EPS))
        _set(V['FSI'][1, md], 0.0)
        _set(V['FSI'][2, md], 0.0)
        n_rr   = sum(1 for d in active if d != md and (md, d) in rr)
        has_rp = (md, 1) in rp
        fr     = 1.0 / max(n_rr + (1 if has_rp else 0), 1)
        n_sr   = sum(1 for d in active if d != md and (md, d) in sr)
        has_sp = (md, 1) in sp
        fs     = 1.0 / max(n_sr + (1 if has_sp else 0), 1)
        for mp in MODS:
            _set(V['split_RR'][md, mp], fr if (md, mp) in rr else 0.0)
            _set(V['split_SR'][md, mp], fs if (md, mp) in sr else 0.0)
        _set(V['split_RP'][md, 1], fr if has_rp else 0.0)
        _set(V['split_SP'][md, 1], fs if has_sp else 0.0)
        # outlet flow values for the warm-start
        for i in COMPS:
            ro = max(f['rN' if i == 1 else 'rO'], EPS)
            so = max(f['pN' if i == 1 else 'pO'], EPS)
            _set(V['FRP'][i, md, 1], ro * fr if has_rp else 0.0)
            _set(V['FSP'][i, md, 1], so * fs if has_sp else 0.0)
            for mp in MODS:
                _set(V['FRR'][i, md, mp], ro * fr if (md, mp) in rr else 0.0)
                _set(V['FSR'][i, md, mp], so * fs if (md, mp) in sr else 0.0)
        # OCFE profile guesses
        for l in ELS:
            for j in PTS:
                p = ((l - 1) + S[j]) / NEL
                fr_total = 0.0
                fs_total = 0.0
                fr_vals = {}
                fs_vals = {}
                for i in COMPS:
                    fi = max(f['inN' if i == 1 else 'inO'], EPS)
                    fo = max(f['rN'  if i == 1 else 'rO'],  EPS)
                    po = max(f['pN'  if i == 1 else 'pO'],  EPS)
                    fr_vals[i] = max(fi * (1 - p) + fo * p, EPS)
                    fs_vals[i] = max(po * p, EPS)
                    _set(V['FR'][i, md, l, j], fr_vals[i])
                    _set(V['FS'][i, md, l, j], fs_vals[i])
                    fr_total += fr_vals[i]
                    fs_total += fs_vals[i]
                fr_total = max(fr_total, EPS)
                fs_total = max(fs_total, EPS)
                _set(V['FRt'][md, l, j], fr_total)
                _set(V['FSt'][md, l, j], fs_total)
                for i in COMPS:
                    xr_v = fr_vals[i] / fr_total
                    xs_v = fs_vals[i] / fs_total
                    _set(V['xR'][i, md, l, j], xr_v)
                    _set(V['xS'][i, md, l, j], xs_v)
                    flux = max(PI[i] * (xr_v * PR - xs_v * PP), 0.0)
                    _set(V['Jf'][i, md, l, j], flux)
def bin(m):
    V = m._V
    for md in MODS:
        for var in [V['y'][md], V['zIR'][1, md], V['zRP'][md, 1], V['zSP'][md, 1]]:
            var.LB = 0.0
            var.UB = 1.0
        for mp in MODS:
            V['zRR'][md, mp].LB = 0.0; V['zRR'][md, mp].UB = 1.0
            V['zSR'][md, mp].LB = 0.0; V['zSR'][md, mp].UB = 1.0

In [7]:
def con(mm=5):
    configs = []
    pool = list(range(1, NMOD + 1))
    for n in range(1, mm + 1):
        for active in combinations(pool, n):
            active = list(active)
            pairs = [(a, b) for a in active for b in active if a != b]
            feed_opts = [list(fo) for r in range(1, n + 1)
                         for fo in combinations(active, r)]
            if n == 1:
                conn_list = [{'rr': set(), 'sr': set()}]
            else:
                rr_sets = [set()]
                for p in pairs:
                    rr_sets = [c | {p} for c in rr_sets] + rr_sets
                sr_sets = [set()]
                conn_list = [{'rr': r, 'sr': s} for r in rr_sets for s in sr_sets]
            for feed_to in feed_opts:
                for conn in conn_list:
                    rr = conn['rr']
                    sr = conn['sr']
                    for rp_bits in range(1, 2**n):
                        rp = set()
                        for idx, md in enumerate(active):
                            if rp_bits & (1 << idx):
                                rp.add((md, 1))
                        sp = set()
                        for md in active:
                            sp.add((md, 1))
                        ok = True
                        for md in active:
                            has_ret_out  = (any((md, d) in rr for d in active if d != md)
                                            or (md, 1) in rp)
                            has_perm_out = (any((md, d) in sr for d in active if d != md)
                                            or (md, 1) in sp)
                            if not has_ret_out or not has_perm_out:
                                ok = False
                                break
                        if not ok:
                            continue
                        areas = {
                            md: (4000 if any((s, md) in rr or (s, md) in sr
                                             for s in active if s != md)
                                 else 2000)
                            for md in active
                        }
                        configs.append({
                            'active':  active,
                            'feed_to': feed_to,
                            'rr':      rr,
                            'sr':      sr,
                            'rp':      rp,
                            'sp':      sp,
                            'areas':   areas
                        })
    return configs
def _val(v):
    try:
        return v.X
    except Exception:
        return 0.0

In [9]:
def verify_physics(m, active):
    V = m._V
    for md in active:
        fri_N = _val(V['FRI'][1, md])
        fri_O = _val(V['FRI'][2, md])
        fro_N = _val(V['FRO'][1, md])
        fro_O = _val(V['FRO'][2, md])
        fri_t = fri_N + fri_O
        fro_t = fro_N + fro_O
        if fri_t < 1.0 or fro_t < 1.0:
            continue
        in_pur  = fri_N / fri_t
        out_pur = fro_N / fro_t
        if out_pur < in_pur - 0.01:
            return False
        if out_pur > 0.999 and in_pur < 0.95:
            return False
        prev_xN = 0.0
        for l in ELS:
            for j in PTS:
                frt = _val(V['FRt'][md, l, j])
                if frt > EPS:
                    xN = _val(V['FR'][1, md, l, j]) / frt
                    if xN < prev_xN - 0.02:
                        return False
                    prev_xN = xN
    return True

In [10]:
def _apply_solver_options(m, time_limit=14400):
    m.Params.NonConvex      = 2          
    m.Params.MIPGap         = 0.001
    m.Params.FeasibilityTol = 1e-6
    m.Params.OptimalityTol  = 1e-6
    m.Params.TimeLimit      = time_limit
    m.Params.Threads        = 0          # use all cores
    m.Params.NumericFocus   = 2          
    m.Params.OutputFlag     = 0

In [11]:
# Solve loop
def solve(m):
    configs = con(mm=5)
    promising = []
    for cfg in configs:
        result = sim_net(
            cfg['active'], cfg['feed_to'],
            cfg['rr'], cfg['sr'],
            cfg['rp'], cfg['sp'],
            cfg['areas'])
        if result and result['pur'] > 0.85 and result['rec'] > 0.35:
            promising.append((cfg, result))
    best_obj = float('inf')
    best_cfg = None
    n_feasible = 0
    t_start = time.time()
    for idx, (cfg, sim_result) in enumerate(promising):
        if time.time() - t_start > 300:
            break
        init(m, cfg['active'], cfg['feed_to'],
             cfg['rr'], cfg['sr'],
             cfg['rp'], cfg['sp'],
             cfg['areas'])
        try:
            _apply_solver_options(m, time_limit=60)
            m.optimize()
            ok_status = m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL,
                                     GRB.TIME_LIMIT, GRB.USER_OBJ_LIMIT)
            has_solution = m.SolCount > 0
            if not (ok_status and has_solution):
                bin(m)
                continue
            FN = m._V['FPR'][1, 1].X
            FO = m._V['FPR'][2, 1].X
            Fp = FN + FO
            if Fp < 1.0:
                bin(m)
                continue
            pur = FN / Fp
            rec = FN / FF[1]
            obj = m.ObjVal
            phys_ok  = verify_physics(m, cfg['active'])
            feasible = pur >= 0.9795 and rec >= 0.4995 and phys_ok
            if feasible:
                n_feasible += 1
                if obj < best_obj:
                    best_obj = obj
                    best_cfg = cfg
        except gp.GurobiError as e:
            print(f"  Gurobi error on cfg {idx}: {e}")
        except Exception:
            pass
        bin(m)
    if best_cfg is None:
        return False
    init(m, best_cfg['active'], best_cfg['feed_to'],
         best_cfg['rr'], best_cfg['sr'],
         best_cfg['rp'], best_cfg['sp'],
         best_cfg['areas'])
    _apply_solver_options(m, time_limit=14400)
    m.Params.OutputFlag = 0     
    m.optimize()
    bin(m)
    return m.SolCount > 0

In [13]:
# Reporting
def report(m, solver_ok):
    V = m._V
    print("\n" + "=" * 92)
    print("  Air Separation Superstructure — Results (Gurobi)")
    print("=" * 92)
    print(f"  Feed: {FT:.0f} kmol/h  ({XF[1]*100:.0f}% N2, {XF[2]*100:.0f}% O2)")
    # Bail out early if no incumbent solution exists.
    if m.SolCount == 0:
        print("\n  No feasible solution found.")
        print(f"  Gurobi status = {m.Status}  "
              f"(3=infeasible, 4=inf_or_unbd, 5=unbounded, 9=time_limit, 11=interrupted)")
        print("=" * 92)
        return
    # Safe access to ObjVal now that SolCount > 0
    obj_val = m.ObjVal
    sep = "-" * 92
    print(f"\n{sep}")
    print(f"  {'Module':10s}  {'Area (m2)':>10}  |  "
          f"{'Ret-in (kmol/h)':>16}  {'N2 frac':>8}  |  "
          f"{'Ret-out (kmol/h)':>16}  {'N2 frac':>8}")
    print(sep)
    active_modules = []
    for md in sorted(MODS):
        if (_val(V['y'][md]) or 0) > 0.5:
            active_modules.append(md)
            area    = _val(V['A'][md])
            fi_tot  = sum(_val(V['FRI'][i, md]) for i in COMPS)
            fo_tot  = sum(_val(V['FRO'][i, md]) for i in COMPS)
            xi_N2   = _val(V['FRI'][1, md]) / fi_tot if fi_tot > 1e-6 else 0.0
            xo_N2   = _val(V['FRO'][1, md]) / fo_tot if fo_tot > 1e-6 else 0.0
            print(f"  Module {md:3d}  {area:10.1f}  |  "
                  f"{fi_tot:16.2f}  {xi_N2:8.4f}  |  "
                  f"{fo_tot:16.2f}  {xo_N2:8.4f}")
    print(sep)
    FN = _val(V['FPR'][1, 1])
    FO = _val(V['FPR'][2, 1])
    Fp = FN + FO
    pur = FN / Fp if Fp > 0 else 0.0
    rec = FN / FF[1]
    total_area = sum(_val(V['A'][md]) for md in active_modules)
    n_modules  = len(active_modules)
    print(f"\n  Product:  purity = {pur:.4f}   recovery = {rec:.4f}")
    print(f"            N2 = {FN:.2f}  O2 = {FO:.2f}  total = {Fp:.2f} kmol/h")
    print(f"  Cost:     {total_area:.0f} m2 x ${CA}/m2 = ${total_area*CA:,.0f}"
          f"  +  {n_modules} modules x ${PHI:.0f} = ${n_modules*PHI:,.0f}"
          f"  =>  total ${obj_val:,.0f}/h")
    for md in active_modules:
        if (_val(V['zIR'][1, md]) or 0) > 0.5:
            ft = sum(_val(V['FIR'][i, 1, md]) for i in COMPS)
            if ft > 0.01:
                print(f"    Feed -> M{md}:  {ft:.2f} kmol/h")
        for mp in MODS:
            if mp != md:
                if (_val(V['zRR'][md, mp]) or 0) > 0.5:
                    ft = sum(_val(V['FRR'][i, md, mp]) for i in COMPS)
                    if ft > 0.01:
                        print(f"    M{md} retentate -> M{mp} inlet:  {ft:.2f} kmol/h")
                if (_val(V['zSR'][md, mp]) or 0) > 0.5:
                    ft = sum(_val(V['FSR'][i, md, mp]) for i in COMPS)
                    if ft > 0.01:
                        print(f"    M{md} permeate -> M{mp} inlet:  {ft:.2f} kmol/h")
        if (_val(V['zRP'][md, 1]) or 0) > 0.5:
            ft = sum(_val(V['FRP'][i, md, 1]) for i in COMPS)
            if ft > 0.01:
                print(f"    M{md} retentate -> product:  {ft:.2f} kmol/h")
        if (_val(V['zSP'][md, 1]) or 0) > 0.5:
            ft = sum(_val(V['FSP'][i, md, 1]) for i in COMPS)
            if ft > 0.01:
                print(f"    M{md} permeate -> waste:  {ft:.2f} kmol/h")
    for md in active_modules:
        vals = []
        for l in ELS:
            for j in PTS:
                frt = _val(V['FRt'][md, l, j])
                xN2 = _val(V['FR'][1, md, l, j]) / frt if frt > EPS else 0.0
                vals.append(f"{xN2:.4f}")
        print(f"    M{md} (A = {_val(V['A'][md]):.0f} m2):  [{', '.join(vals)}]")
    pN = _val(V['FPR'][1, 1])
    pO = _val(V['FPR'][2, 1])
    sN = _val(V['FPS'][1, 1])
    sO = _val(V['FPS'][2, 1])
    mass_bal_err = abs(pN + sN - FF[1]) + abs(pO + sO - FF[2])
    print(f"\n  Mass balance:  feed = {FT:.0f}  product = {pN+pO:.2f}"
          f"  waste = {sN+sO:.2f}  error = {mass_bal_err:.4f} kmol/h")
    status = ("Solver confirmed optimality" if solver_ok
              else "Check solution")
    print(f"  Status: {status}  (Gurobi status code = {m.Status})")
    print("=" * 92)
# Main
def main():
    print("=" * 72)
    print("Air Separation Superstructure MINLP  (Gurobi backend)")
    print(f"  {NMOD} module slots  |  OCFE: {NEL} elements x {NCP} points")
    print(f"  Feed: {FT:.0f} kmol/h")
    print(f"  Objective: minimize CA*A + PHI*y  (CA=${CA}/m2, PHI=${PHI}/module)")
    print("=" * 72)

    m = build_model()

    n_vars = m.NumVars
    n_cons = m.NumConstrs + m.NumQConstrs
    n_bin  = m.NumBinVars
    print(f"\n  Model size: {n_vars} variables, {n_cons} constraints "
          f"({m.NumQConstrs} quadratic)")

    solver_ok = solve(m)
    report(m, solver_ok)
    return m


if __name__ == "__main__":
    m = main()

Air Separation Superstructure MINLP  (Gurobi backend)
  5 module slots  |  OCFE: 2 elements x 4 points
  Feed: 2300 kmol/h
  Objective: minimize CA*A + PHI*y  (CA=$20.0/m2, PHI=$1000.0/module)
Set parameter Username
Set parameter LicenseID to value 2817105
Academic license - for non-commercial use only - expires 2027-05-01

  Model size: 919 variables, 1738 constraints (620 quadratic)

  Air Separation Superstructure — Results (Gurobi)
  Feed: 2300 kmol/h  (79% N2, 21% O2)

--------------------------------------------------------------------------------------------
  Module       Area (m2)  |   Ret-in (kmol/h)   N2 frac  |  Ret-out (kmol/h)   N2 frac
--------------------------------------------------------------------------------------------
  Module   1      4136.4  |           1546.51    0.9383  |           1119.05    0.9800
  Module   2      4450.2  |           2300.00    0.7900  |           1546.51    0.9383
--------------------------------------------------------------------------